# Buổi 9: Ứng dụng tổng hợp và báo cáo đồ án
**Môn học:** Python cho Phân tích Dữ liệu Kinh doanh · **BÀI TẬP THỰC HÀNH**

**Mục tiêu:**
- Đi trọn quy trình một dự án phân tích kinh doanh: câu hỏi → dữ liệu → phân tích → mô hình → khuyến nghị
- Kết hợp các kỹ năng của buổi 1–8 trong một bài toán
- Trình bày kết quả và khuyến nghị cho nhà quản lý

**Dữ liệu dùng trong buổi:** `ecommerce_orders_clean.csv`, `customers.csv`, `supermarket_sales.csv` (trong thư mục `data/`). Nếu dùng Google Colab, tải thư mục `data/` lên rồi sửa biến `DATA` bên dưới.

**Cấu trúc:** Phần A – bài tập có hướng dẫn (làm theo, chạy từng ô) · Phần B – bài tập tự làm (bắt buộc, nộp cuối buổi) · Phần C – bài tập làm thêm (dành cho bạn muốn thử thách).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, mean_absolute_error
DATA = "../data/"
sns.set_theme(style="whitegrid")

## Phần A. Bài tập có hướng dẫn – Tình huống mẫu

**Bối cảnh:** Một công ty thương mại điện tử chuẩn bị ngân sách khuyến mãi quý 4/2025. Ban giám đốc hỏi:
1. Tình hình kinh doanh 9 tháng đầu năm thế nào? Nhóm hàng, kênh nào dẫn dắt?
2. Khách hàng nào có khả năng **mua lại trong quý 4**, khách nào cần kích hoạt?
3. Nên gửi ưu đãi cho ai để chi phí hiệu quả nhất?

Ta dùng dữ liệu đến hết tháng 9 để xây dựng đặc trưng, và dữ liệu quý 4 để biết khách có mua lại hay không.

### A1. Bước 1 – Chuẩn bị dữ liệu

In [ ]:
od = pd.read_csv(DATA + "ecommerce_orders_clean.csv", parse_dates=["ngay_dat"])
kh = pd.read_csv(DATA + "customers.csv", parse_dates=["ngay_dang_ky"])
od = od[od["trang_thai"] == "Hoàn thành"].copy()
od["doanh_thu"] = od["so_luong"] * od["don_gia"]

moc = pd.Timestamp("2025-10-01")
truoc = od[od["ngay_dat"] < moc]          # dữ liệu dùng để mô tả hành vi
sau = od[od["ngay_dat"] >= moc]           # dữ liệu dùng để tạo nhãn
print(len(truoc), "đơn trước quý 4;", len(sau), "đơn trong quý 4")

### A2. Bước 2 – Phân tích mô tả (trả lời câu hỏi 1)

In [ ]:
tong = truoc["doanh_thu"].sum() / 1e9
print(f"Doanh thu 9 tháng: {tong:.1f} tỷ đồng – {truoc['ma_don'].nunique()} đơn – {truoc['ma_kh'].nunique()} khách")

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
(truoc.set_index("ngay_dat")["doanh_thu"].resample("ME").sum() / 1e9).plot(ax=ax[0], marker="o")
ax[0].set(title="Doanh thu theo tháng (tỷ đồng)", xlabel="")
(truoc.groupby("nhom_hang")["doanh_thu"].sum().sort_values() / 1e9).plot(kind="barh", ax=ax[1], color="#1f6feb")
ax[1].set(title="Theo nhóm hàng (tỷ đồng)", ylabel="")
(truoc.groupby("kenh")["doanh_thu"].sum().sort_values() / 1e9).plot(kind="barh", ax=ax[2], color="#2da44e")
ax[2].set(title="Theo kênh (tỷ đồng)", ylabel="")
plt.tight_layout(); plt.show()

### A3. Bước 3 – Xây dựng đặc trưng khách hàng và nhãn

In [ ]:
dac_trung = truoc.groupby("ma_kh").agg(
    recency=("ngay_dat", lambda x: (moc - x.max()).days),
    so_don=("ma_don", "nunique"),
    tong_chi=("doanh_thu", "sum"),
    aov=("doanh_thu", "mean"),
    so_nhom_hang=("nhom_hang", "nunique"),
    ty_le_app=("kenh", lambda x: (x == "App").mean()),
    ngay_dau=("ngay_dat", "min"),
)
dac_trung["thang_gan_bo"] = ((moc - dac_trung["ngay_dau"]).dt.days / 30).round(1)
dac_trung = dac_trung.drop(columns="ngay_dau").join(kh.set_index("ma_kh")[["tuoi", "vung"]])
dac_trung["mua_lai_q4"] = dac_trung.index.isin(sau["ma_kh"]).astype(int)
print("Số khách:", len(dac_trung), "– tỷ lệ mua lại Q4:", dac_trung["mua_lai_q4"].mean().round(3))
dac_trung.head()

### A4. Bước 4 – Mô hình dự đoán khả năng mua lại (trả lời câu hỏi 2)

In [ ]:
X = pd.get_dummies(dac_trung.drop(columns="mua_lai_q4"), columns=["vung"], drop_first=True, dtype=int)
y = dac_trung["mua_lai_q4"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

sc = StandardScaler().fit(X_tr)
lg = LogisticRegression(max_iter=1000).fit(sc.transform(X_tr), y_tr)
rf = RandomForestClassifier(n_estimators=400, max_depth=6, random_state=42).fit(X_tr, y_tr)

for ten, p in [("Logistic", lg.predict_proba(sc.transform(X_te))[:, 1]), ("Random Forest", rf.predict_proba(X_te)[:, 1])]:
    d = (p >= 0.5).astype(int)
    print(f"{ten:14} AUC={roc_auc_score(y_te, p):.3f}  F1={f1_score(y_te, d):.3f}  "
          f"Precision={precision_score(y_te, d):.3f}  Recall={recall_score(y_te, d):.3f}")

**AUC** (0,5 = đoán mò, 1 = hoàn hảo) đo khả năng xếp hạng khách theo xác suất – phù hợp khi ta muốn chọn top khách để gửi ưu đãi.

In [ ]:
pd.Series(rf.feature_importances_, index=X.columns).sort_values().plot(kind="barh", title="Yếu tố dự báo mua lại")
plt.show()

### A5. Bước 5 – Phân nhóm hành động (trả lời câu hỏi 3)

In [ ]:
dac_trung["xac_suat"] = rf.predict_proba(X)[:, 1]
gia_tri_cao = dac_trung["tong_chi"] >= dac_trung["tong_chi"].median()
kha_nang_cao = dac_trung["xac_suat"] >= 0.5
dac_trung["nhom_hanh_dong"] = np.select(
    [gia_tri_cao & kha_nang_cao, gia_tri_cao & ~kha_nang_cao, ~gia_tri_cao & kha_nang_cao],
    ["Giữ quan hệ (không cần giảm giá)", "Ưu tiên ưu đãi giữ chân", "Gợi ý mua kèm"],
    default="Ưu đãi chi phí thấp / email")
bang = dac_trung.groupby("nhom_hanh_dong").agg(so_kh=("so_don", "size"), xac_suat_tb=("xac_suat", "mean"),
                                               chi_tieu_tb_trieu=("tong_chi", lambda x: x.mean() / 1e6))
bang.round(2)

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=dac_trung, x="xac_suat", y=dac_trung["tong_chi"] / 1e6, hue="nhom_hanh_dong", s=20)
plt.axvline(0.5, ls="--", c="gray"); plt.axhline(dac_trung["tong_chi"].median() / 1e6, ls="--", c="gray")
plt.yscale("log"); plt.ylabel("Tổng chi 9 tháng (triệu đồng, thang log)"); plt.xlabel("Xác suất mua lại Q4")
plt.title("Ma trận hành động khách hàng"); plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left"); plt.show()

### A6. Bước 6 – Khuyến nghị cho ban giám đốc
Một khuyến nghị tốt có **kết luận + số liệu + hành động + cách đo lường**. Ví dụ:

> *"Nhóm 'Ưu tiên ưu đãi giữ chân' gồm khách chi tiêu cao nhưng xác suất mua lại thấp. Đề xuất dành phần lớn ngân sách
> khuyến mãi quý 4 cho nhóm này (voucher cá nhân hóa theo nhóm hàng đã mua). Đo lường bằng tỷ lệ mua lại của nhóm được gửi
> ưu đãi so với một nhóm đối chứng 10% không được gửi."*

In [ ]:
dac_trung.sort_values("xac_suat").to_csv("danh_sach_hanh_dong_Q4.csv", encoding="utf-8-sig")
print("Đã xuất danh sách cho bộ phận marketing.")

## Phần B. Bài tập tự làm – Khung đồ án nhóm

Phần này là **khung đồ án**. Mỗi nhóm thay dữ liệu và câu hỏi theo đề tài đã đăng ký. Nếu chưa có dữ liệu riêng,
hãy luyện tập với `supermarket_sales.csv` và câu hỏi *làm thế nào tăng lợi nhuận gộp cho chuỗi siêu thị?*
(lời giải tham khảo của giảng viên dùng đúng dữ liệu và câu hỏi này).
Nộp: notebook chạy được từ đầu đến cuối + báo cáo 8–12 trang + slide thuyết trình 10–12 phút.

### Bài B1. Câu hỏi kinh doanh và dữ liệu

Viết (trong ô Markdown) 2–3 câu hỏi kinh doanh cụ thể, đo lường được. Đọc dữ liệu, mô tả nguồn, số dòng, số cột,
ý nghĩa các cột chính và giai đoạn dữ liệu.

In [ ]:
# Viết code của bạn ở đây


### Bài B2. Làm sạch và chuẩn bị dữ liệu

Kiểm tra thiếu, trùng, kiểu dữ liệu, ngoại lai; tạo các cột phái sinh cần cho phân tích (tháng, thứ, khung giờ, biên lợi nhuận...).
Ghi lại mọi quyết định làm sạch và lý do.

In [ ]:
# Viết code của bạn ở đây


### Bài B3. Phân tích khám phá và trực quan hóa

Ít nhất 4 biểu đồ trả lời trực tiếp các câu hỏi ở B1, mỗi biểu đồ kèm 1–2 câu nhận xét có số liệu.

In [ ]:
# Viết code của bạn ở đây


### Bài B4. Mô hình học máy

Chọn ít nhất một mô hình phù hợp với câu hỏi (hồi quy, phân lớp hoặc phân cụm). So sánh tối thiểu hai mô hình/cấu hình,
đánh giá bằng chỉ số phù hợp và diễn giải kết quả bằng ngôn ngữ kinh doanh.

In [ ]:
# Viết code của bạn ở đây


### Bài B5. Khuyến nghị và kế hoạch hành động

Viết 3 khuyến nghị theo cấu trúc *kết luận – bằng chứng (số liệu) – hành động – chỉ số đo lường*.
Nêu hạn chế của phân tích và hướng mở rộng.

In [ ]:
# Viết code của bạn ở đây


## Phần C. Bài tập làm thêm

### Bài C1. Báo cáo tự động bằng HTML

Viết hàm `xuat_bao_cao(df, ten_tep)` tạo một tệp HTML gồm tiêu đề, bảng KPI (`DataFrame.to_html()`) và 2 biểu đồ
(lưu PNG rồi nhúng bằng thẻ `<img>`). Mở tệp bằng trình duyệt để kiểm tra.

In [ ]:
# Viết code của bạn ở đây


### Bài C2. Ước lượng hiệu quả chiến dịch bằng mô phỏng

Dùng `dac_trung` ở Phần A: giả sử gửi voucher 100.000 đ cho nhóm "Ưu tiên ưu đãi giữ chân", voucher làm xác suất mua lại tăng
thêm 15 điểm phần trăm (tối đa 1), mỗi lần mua lại mang lại lợi nhuận bằng 15% giá trị đơn trung bình (`aov`) của khách.
Ước tính lợi nhuận tăng thêm trừ chi phí voucher (chỉ tính voucher được dùng = khách mua lại). Chiến dịch có đáng làm không?

In [ ]:
# Viết code của bạn ở đây


### Bài C3. Ứng dụng web nhỏ với Streamlit

Viết tệp `app.py` dùng Streamlit cho phép chọn thành phố và nhóm hàng, hiển thị KPI và biểu đồ doanh thu theo tháng.
Chạy bằng lệnh `streamlit run app.py` trên máy cá nhân (`pip install streamlit`). Ô dưới chỉ ghi tệp, không chạy ứng dụng.

In [ ]:
# Viết code của bạn ở đây
